# SP519 · Prédiction du flux d'électrons de 1 MeV en LEO (NOAA-15)

## 3. Step 1 : préparation des données et analyse statistique
Le notebook suit, dans l'ordre, les 7 points du Step 1 de l'énoncé :

* Récupérer les données NOAA-15 SEM-2 et les paramètres solaires pour 2020–2024
* Nettoyer et synchroniser les données (valeurs manquantes, valeurs invalides, outliers, horodatage)
* Choisir la région des ceintures de radiation (plage de L) et justifier ce choix
* Moyenner tous les paramètres sur des intervalles de 12 h
* Visualiser les séries temporelles (flux vs temps, L vs temps, paramètres solaires vs temps)
* Statistiques descriptives, relations entre le flux et les paramètres solaires, outliers
* Déterminer le seuil du 95ᵉ percentile du flux, qui définit les « événements »

(La première cellule de code ci-dessous contient seulement les imports et les paramètres.)

## Imports et paramètres

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.0, 'font.size': 9})

# Dossiers (relatifs au notebook)
PROJ = os.getcwd()
DATA = os.path.join(PROJ, 'data')
FIG = os.path.join(PROJ, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Paramètres de l'étude
START, END = pd.Timestamp('2020-01-01'), pd.Timestamp('2025-01-01')   # période [START, END[
STEP = '12h'            # résolution temporelle des prédictions
L_LO, L_HI = 3.0, 7.0   # région de la ceinture prédite : L_LO <= L < L_HI
P_EVENT = 95            # percentile définissant un "événement"

def savefig(fig, name):
    # Sauvegarde des figures pour le rapport
    fig.savefig(os.path.join(FIG, name), dpi=150, bbox_inches='tight')

print('Dossier du projet :', PROJ)

## Récupération des données (NOAA-15 SEM-2 + paramètres solaires, 2020–2024)

### Téléchargement des fichiers bruts NOAA-15
Un fichier par jour sur le serveur NCEI (`poes_n15_AAAAMMJJ_proc.nc`, ~6,6 Mo, une mesure toutes les 2 s), soit
~1 800 fichiers et ~12 Go pour 2020–2024. Ils sont rangés dans `data/noaa15_raw/` (ignoré par Git).
Le téléchargement est **désactivé par défaut** (`TELECHARGER = False`) pour que le notebook puisse s'exécuter de haut
en bas sans retélécharger 12 Go : les tables résumées sont déjà dans `data/`.

In [ ]:
import calendar
from concurrent.futures import ThreadPoolExecutor
from typing import List

import requests

TELECHARGER = False                          # True pour (re)télécharger les fichiers bruts NOAA-15
RAW_DIR = os.path.join(DATA, 'noaa15_raw')   # fichiers bruts (ignorés par Git)
MAX_WORKERS = 4                              # au-delà, le serveur NCEI bloque des connexions

def download_nc_files(
    years: List[int],
    months: List[int],
    data_dir: str = RAW_DIR,
    max_workers: int = MAX_WORKERS,
) -> None:
    """Downloads NOAA-15 POES daily NetCDF files into data_dir.

    Files already present are skipped (the download can be resumed), and a file
    whose download fails is deleted (no half-written file is left behind).

    Args:
        years: List of years (e.g., [2020, 2021]).
        months: List of months as integers; downloads every day in each month.
        data_dir: Path to the local directory where files will be stored.
        max_workers: Maximum number of files to download concurrently.
    """
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1")

    files_to_download = []
    for year in years:
        for month in months:
            if not 1 <= month <= 12:
                raise ValueError(f"Month must be between 1 and 12, got {month}")
            for day in range(1, calendar.monthrange(year, month)[1] + 1):
                date_str = f"{year:04d}{month:02d}{day:02d}"
                files_to_download.append(
                    (
                        year,
                        date_str,
                        f"poes_n15_{date_str}_proc.nc",
                    )
                )

    # 1. Create the destination directory (existing files are kept)
    os.makedirs(data_dir, exist_ok=True)

    base_url = (
        "https://www.ncei.noaa.gov/data/poes-metop-space-environment-monitor/"
        "access/l1b/v01r00/{year}/noaa15/poes_n15_{date_str}_proc.nc"
    )

    def download_file(file_info):
        year, date_str, filename = file_info
        file_url = base_url.format(year=year, date_str=date_str)
        destination_path = os.path.join(data_dir, filename)
        if os.path.exists(destination_path):
            return                                   # already downloaded
        tmp_path = destination_path + ".part"        # renamed only once complete

        print(f"Downloading: {file_url}")
        try:
            with requests.get(file_url, stream=True, timeout=30) as response:
                response.raise_for_status()

                with open(tmp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

            os.replace(tmp_path, destination_path)
            print(f"Successfully saved to {destination_path}")

        except requests.exceptions.RequestException as e:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            print(f"Failed to download {file_url}: {e}")

    # 2. Download daily files concurrently to reduce time spent waiting on the server.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(download_file, files_to_download))


if TELECHARGER:
    download_nc_files(years=list(range(START.year, END.year)), months=list(range(1, 13)))
else:
    print('Téléchargement désactivé (TELECHARGER = False) : on utilise les tables déjà présentes dans data/.')

### Paramètres solaires et géomagnétiques (OMNI)
Base **OMNI2 horaire** (`OMNI2_H0_MRG1HR`) de la NASA (SPDF / CDAWeb) : mesures du vent solaire (sondes ACE, Wind…)
ramenées jusqu'à la Terre, et indices géomagnétiques, sur une même base de temps horaire UTC. Deux fichiers CDF par an
(janvier–juin, juillet–décembre), soit 10 fichiers d'environ 1,5 Mo, lus avec `cdflib`.

| Variable OMNI | Nom dans le notebook | Rôle physique |
|---|---|---|
| `V` | `V_kms` | vitesse du vent solaire : principal moteur de l'accélération des électrons relativistes |
| `BZ_GSM` | `Bz_nT` | Bz < 0 → reconnexion magnétique, injection de particules dans la magnétosphère |
| `F10_INDEX` | `F107_sfu` | flux radio solaire à 10,7 cm : niveau d'activité du cycle solaire |
| `N`, `Pressure` | `N_cm3`, `P_nPa` | densité et pression dynamique : compression de la magnétosphère, pertes d'électrons |
| `E` | `E_mVm` | champ électrique −V·Bz : couplage vent solaire / magnétosphère |
| `ABS_B` | `B_nT` | intensité du champ magnétique interplanétaire |
| `KP`, `DST`, `AE` | `Kp`, `Dst_nT`, `AE_nT` | activité géomagnétique : globale (Kp), orages (Dst), sous-orages (AE) |

V, Bz et F10.7 sont demandés par l'énoncé ; les autres sont ajoutés pour les raisons physiques ci-dessus.
Les valeurs manquantes sont codées dans les CDF par une valeur de remplissage (attribut `FILLVAL`, ex. 9999 pour V) :
on les remplace par NaN. `KP` est stocké ×10 (Kp = 3+ → 33) : on le divise par 10.
Le flux de protons > 10 MeV (`PR-FLX_10`) n'est pas gardé : il manque à ~97 % sur 2020–2024.

In [ ]:
import cdflib

OMNI_DIR = os.path.join(DATA, 'omni')   # fichiers CDF bruts (ignorés par Git)
OMNI_URL = 'https://spdf.gsfc.nasa.gov/pub/data/omni/omni_cdaweb/hourly/{y}/omni2_h0_mrg1hr_{y}{m}01_v01.cdf'
OMNI_VARS = {'V': 'V_kms', 'BZ_GSM': 'Bz_nT', 'F10_INDEX': 'F107_sfu', 'N': 'N_cm3', 'Pressure': 'P_nPa',
             'E': 'E_mVm', 'ABS_B': 'B_nT', 'KP': 'Kp', 'DST': 'Dst_nT', 'AE': 'AE_nT'}
os.makedirs(OMNI_DIR, exist_ok=True)

morceaux = []
for y in range(START.year, END.year):
    for m in ('01', '07'):                                   # un fichier par semestre
        path = os.path.join(OMNI_DIR, f'omni2_h0_mrg1hr_{y}{m}01_v01.cdf')
        if not os.path.exists(path):                         # téléchargé une seule fois
            r = requests.get(OMNI_URL.format(y=y, m=m), timeout=60)
            r.raise_for_status()
            with open(path, 'wb') as f:
                f.write(r.content)
        cdf = cdflib.CDF(path)
        d = {'time': cdflib.cdfepoch.to_datetime(cdf.varget('Epoch'))}
        for var, nom in OMNI_VARS.items():
            x = cdf.varget(var).astype(float)
            fill = cdf.varattsget(var)['FILLVAL']
            x[np.isclose(x, fill) | (x >= fill)] = np.nan       # valeur de remplissage -> NaN
            d[nom] = x
        morceaux.append(pd.DataFrame(d))

omni_raw = pd.concat(morceaux).drop_duplicates('time').set_index('time').sort_index()
omni_raw = omni_raw.loc[START:END - pd.Timedelta('1h')]
omni_raw['Kp'] = omni_raw['Kp'] / 10                         # Kp*10 -> Kp
omni_raw.index = pd.DatetimeIndex(omni_raw.index).as_unit('ns')
omni_raw.to_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))
print('OMNI :', omni_raw.shape, '|', omni_raw.index.min(), '->', omni_raw.index.max())

### Tables utilisées dans la suite du notebook
La cellule ci-dessous lit les deux tables de `data/` ; tout le reste du notebook ne dépend que d'elles.

* `noaa15_p6_hourly_Lbins.parquet` : le flux `mep_pro_tel90_flux_p6` de NOAA-15, résumé **heure par heure et par
  tranche de L** (L de 1 à 8 par pas de 0,25, et `Lbin = 10` pour la calotte polaire L ≥ 10).
  Colonnes : `time` (début de l'heure, UTC), `Lbin`, `p6_sum` (somme des flux), `n` (nombre de mesures),
  `p6_max` (plus grande mesure de 2 s de la case, utile pour repérer les pics aberrants),
  `omni_p1_sum` / `omni_p1_n` (idem pour les protons 25 MeV, utiles au nettoyage).
  → Produite une fois à partir des fichiers bruts, hors notebook (~12 Go à traiter).
* `omni_hourly_2020_2024.parquet` : paramètres OMNI heure par heure (index `time`, UTC) : `V_kms`, `Bz_nT`,
  `F107_sfu`, `N_cm3`, `P_nPa`, `E_mVm`, `B_nT`, `Kp`, `Dst_nT`, `AE_nT`.
  → Produite par la cellule OMNI ci-dessus.

In [ ]:
noaa = pd.read_parquet(os.path.join(DATA, 'noaa15_p6_hourly_Lbins.parquet'))
omni = pd.read_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))

L_POLAR = 10.0   # code de la tranche "calotte polaire" (L >= 10)

print('NOAA-15 :', noaa.shape, '|', noaa.time.min(), '->', noaa.time.max())
print('   tranches de L :', [float(l) for l in sorted(noaa.Lbin.unique())])
jours = noaa.time.dt.floor('D').unique()
print('   jours avec données :', len(jours), '/', (END - START).days)
print('OMNI    :', omni.shape, '|', omni.index.min(), '->', omni.index.max())
print('   % manquant :', omni.isna().mean().mul(100).round(2).to_dict())
noaa.head()

## Nettoyage et synchronisation

Le nettoyage se fait à deux niveaux (code écrit par l'autre membre du binôme, intégré ici) :

* **les mesures brutes** (une toutes les 2 s, un fichier par jour). Démonstration sur une journée de ce
  qu'il faut retirer d'un fichier brut. La réduction des ~1 800 fichiers bruts en table horaire par tranche de L
  (`noaa15_p6_hourly_Lbins.parquet`) a été faite une fois, hors notebook (~12 Go à télécharger).
* **la table horaire** utilisée par toute la suite : heures sans mesure, pics aberrants, périodes où P6 ne
  mesure plus des électrons (protons solaires), et synchronisation avec OMNI.

Ce que fait le code ci-dessous sur les mesures brutes :
* **horodatage** (`as_dataframe`) : temps converti en dates UTC, triées, sans doublon ;
* **valeurs invalides / manquantes** (`clean_raw`) : flux négatifs (valeurs de remplissage −999) → NaN ; L invalide
  (−1 = valeur de remplissage, L ≈ 100 = ligne de champ ouverte) ; mesures prises pendant l'étalonnage en vol
  (`mep_IFC_on = 1`) retirées ; mesures sans flux ou sans L retirées ;
* **valeurs aberrantes** (`mask_outliers`) : dans chaque tranche de L de largeur 1, une mesure est aberrante si son
  log10(flux) sort de [Q1 − 3·IQR ; Q3 + 3·IQR]. Limite constatée : dans les zones presque vides (« slot »,
  calotte polaire), cette règle retire de vraies mesures de 2 à 4 coups. C'est pourquoi les pics aberrants sont
  traités autrement dans le nettoyage de la table horaire (seuil physique) ;
* **synchronisation** (`synchronize`) : met d'autres séries (OMNI : V, Bz, F10.7…) sur les dates NOAA-15 (valeur la
  plus proche, à moins d'1 h).

### Fonctions de nettoyage des mesures brutes

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

TARGET = "mep_pro_tel90_flux_p6"
L_COL = "L_IGRF"
P6_COUNT_STEP = 243.9024  # flux of one count in the P6 channel [#/cm2-s-str]
L_VALID_RANGE = (1.0, 20.0)  # L outside is a fill value (-1) or an open field line

RAW_COLUMNS = (TARGET, "mep_pro_tel0_flux_p6", "mep_ele_tel90_flux_e3", L_COL, "MLT",
               "mag_lat_sat", "lat", "lon", "alt", "Btot_sat", "mep_IFC_on")


@dataclass
class CleanConfig:
    target: str = TARGET
    l_col: str = L_COL
    drop_ifc: bool = True  # drop samples taken during in-flight calibration (flag == 1)
    outlier_k: float | None = 3.0  # IQR multiplier on log10 flux (None = keep outliers)
    outlier_l_bin: float = 1.0  # outliers are searched inside L bins of this width
    outlier_min_group: int = 50  # minimum positive samples in a bin to apply the rule


# --------------------------------------------------------------------------------------
# Time stamps
# --------------------------------------------------------------------------------------
def _to_datetime_index(t) -> pd.DatetimeIndex:
    t = pd.Series(np.asarray(t))
    if pd.api.types.is_datetime64_any_dtype(t):
        return pd.DatetimeIndex(t)
    v = t.astype("int64")  # NOAA 'time' is uint64 milliseconds since 1970
    m = v.abs().max()
    unit = "ns" if m > 1e17 else "us" if m > 1e14 else "ms" if m > 1e11 else "s"
    return pd.DatetimeIndex(pd.to_datetime(v, unit=unit))


def as_dataframe(data, columns: Iterable[str] | None = None) -> pd.DataFrame:
    """Loader chunk -> float64 DataFrame indexed by naive-UTC, sorted, unique time stamps.

    Accepts a DataFrame, an xarray.Dataset, a dict of arrays or a NetCDF path. For xarray the
    variables are read one by one (``columns`` defaults to ``RAW_COLUMNS``): in the NCEI files
    every variable is also its own dimension, so ``Dataset.to_dataframe()`` would explode.
    """
    if isinstance(data, (str, os.PathLike)):
        import xarray as xr

        data = xr.open_dataset(data, decode_times=False)

    if isinstance(data, pd.DataFrame):
        df = data.copy()
        if columns is not None:
            keep = [c for c in columns if c in df.columns]
            df = df[keep + (["time"] if "time" in df.columns and "time" not in keep else [])]
    elif hasattr(data, "variables") and hasattr(data, "attrs"):  # xarray.Dataset
        names = [c for c in (columns or RAW_COLUMNS) if c in data.variables]
        df = pd.DataFrame({n: np.asarray(data[n].values) for n in names})
        if "time" in data.variables:
            df["time"] = np.asarray(data["time"].values)
    else:
        df = pd.DataFrame({k: np.asarray(v) for k, v in dict(data).items()})
        if columns is not None:
            df = df[[c for c in [*columns, "time"] if c in df.columns]]

    if isinstance(df.index, pd.DatetimeIndex):
        idx = df.index
    elif "time" in df.columns:
        idx = _to_datetime_index(df.pop("time").to_numpy())
    else:
        raise KeyError("no DatetimeIndex and no 'time' column in the data")
    if idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    df.index = pd.DatetimeIndex(idx, name="time")
    return df.astype("float64")


# --------------------------------------------------------------------------------------
# Invalid / missing values and outliers
# --------------------------------------------------------------------------------------
def mask_outliers(df: pd.DataFrame, cfg: CleanConfig) -> pd.Series:
    """Boolean mask of outliers of the target: IQR rule on log10 flux, per L bin.

    Zero-count samples are real values and are neither used for the quartiles nor flagged.
    Bins with fewer than ``outlier_min_group`` positive samples are left untouched.
    """
    out = pd.Series(False, index=df.index)
    if cfg.outlier_k is None:
        return out
    flux = df[cfg.target]
    logv = np.log10(flux.where(flux > 0))
    lbin = np.floor(df[cfg.l_col] / cfg.outlier_l_bin)
    g = logv.groupby(lbin)
    q = g.quantile([0.25, 0.75]).unstack()
    n = g.count()
    ok = n >= cfg.outlier_min_group
    q1, q3 = lbin.map(q[0.25].where(ok)), lbin.map(q[0.75].where(ok))
    iqr = q3 - q1
    return ((logv < q1 - cfg.outlier_k * iqr) | (logv > q3 + cfg.outlier_k * iqr)).fillna(False)


def clean_raw(df: pd.DataFrame, cfg: CleanConfig | None = None) -> tuple[pd.DataFrame, dict]:
    """Remove invalid and missing values, flag outliers. Returns ``(clean_df, report)``."""
    cfg = cfg or CleanConfig()
    for c in (cfg.target, cfg.l_col):
        if c not in df.columns:
            raise KeyError(f"column {c!r} missing (got {list(df.columns)})")
    df = df.copy()
    rep = {"n_raw": len(df),
           "duplicate_timestamps": int(df.index.duplicated().sum()),
           "unsorted_timestamps": int((df.index.to_series().diff() < pd.Timedelta(0)).sum())}
    df = df[~df.index.duplicated(keep="first")].sort_index()

    # fill values (-999 in several MEPED channels) are negative fluxes -> NaN
    for c in [c for c in df.columns if "flux" in c]:
        bad = df[c] < 0
        if bad.any():
            rep[f"fill_values_{c}"] = int(bad.sum())
            df.loc[bad, c] = np.nan

    # invalid L: -1 fill value, open field lines (L ~ 100)
    bad_l = ~df[cfg.l_col].between(*L_VALID_RANGE)
    rep["invalid_L"] = int(bad_l.sum())
    df.loc[bad_l, cfg.l_col] = np.nan

    # in-flight calibration (flag 1 = on; -1 is "not applicable" and kept)
    if cfg.drop_ifc and "mep_IFC_on" in df.columns:
        ifc = df["mep_IFC_on"] == 1
        rep["ifc_on"] = int(ifc.sum())
        df = df[~ifc]

    # missing values: a sample is useless without target or L
    miss = df[[cfg.target, cfg.l_col]].isna().any(axis=1)
    rep["rows_removed_missing"] = int(miss.sum())
    df = df[~miss]

    # outliers -> NaN on the target, then drop those rows
    out = mask_outliers(df, cfg)
    rep["outliers"] = int(out.sum())
    df = df[~out]
    rep["n_clean"] = len(df)
    return df, rep


def clean(data, cfg: CleanConfig | None = None, columns=None) -> tuple[pd.DataFrame, dict]:
    """``as_dataframe`` + ``clean_raw``."""
    return clean_raw(as_dataframe(data, columns), cfg)


# --------------------------------------------------------------------------------------
# Synchronisation
# --------------------------------------------------------------------------------------
def synchronize(base: pd.DataFrame, *others: pd.DataFrame, tolerance: str = "1h") -> pd.DataFrame:
    """Attach other series (e.g. OMNI V, Bz, F10.7) to the NOAA time stamps.

    Each value is the nearest one in time, accepted only within ``tolerance`` (else NaN).
    ``others`` go through ``as_dataframe`` so they can come in any format / time unit.
    """
    out = base.copy()
    tol = pd.Timedelta(tolerance)
    for o in others:
        o = as_dataframe(o)
        o = o[~o.index.duplicated(keep="first")].sort_index()
        o = o.drop(columns=[c for c in o.columns if c in out.columns])
        out = out.join(o.reindex(out.index, method="nearest", tolerance=tol))
    return out


# --------------------------------------------------------------------------------------
# Saving
# --------------------------------------------------------------------------------------
def save_cleaned(df: pd.DataFrame, outdir: str = "data/cleaned", name: str | None = None) -> str | None:
    """Save the cleaned data in ``outdir`` (created if needed) ONLY if the folder is empty.

    Parquet if pyarrow/fastparquet is installed, otherwise gzipped CSV. Returns the written path,
    or None when the folder already contains files (nothing is overwritten).
    """
    os.makedirs(outdir, exist_ok=True)
    if os.listdir(outdir):
        print(f"{outdir} is not empty: nothing saved")
        return None
    name = name or f"cleaned_{df.index[0]:%Y%m%d}_{df.index[-1]:%Y%m%d}"
    try:
        path = os.path.join(outdir, name + ".parquet")
        df.to_parquet(path)
    except ImportError:
        path = os.path.join(outdir, name + ".csv.gz")
        df.to_csv(path)
    print(f"saved {len(df)} rows to {path}")
    return path

### Test sur une journée brute
On applique le nettoyage au fichier brut du 01/01/2020 (`data/noaa15_raw/`, non versionné : la cellule ne fait rien
s'il est absent). Le fichier est lu avec `h5netcdf` : netCDF4 est bloqué par Windows et ne sait pas ouvrir un chemin
contenant un accent (« Donnée »).

In [ ]:
import xarray as xr

def lire_jour_noaa(path, columns=RAW_COLUMNS):
    """Lit un fichier brut NOAA-15 (1 jour, une mesure toutes les 2 s).
    decode_times=False : on garde les nombres bruts, as_dataframe convertit le temps lui-même."""
    with xr.open_dataset(path, engine='h5netcdf', decode_times=False, decode_timedelta=False) as ds:
        return as_dataframe(ds, columns)

fichier_test = os.path.join(RAW_DIR, 'poes_n15_20200101_proc.nc')
if os.path.exists(fichier_test):
    brut = lire_jour_noaa(fichier_test)
    propre, rapport = clean_raw(brut, CleanConfig())
    print(f'{len(brut)} mesures brutes -> {len(propre)} mesures propres')
    for k, v in rapport.items():
        print(f'  {k:<25}{v:>8}')
else:
    print(f'Fichier brut absent ({fichier_test}) : test ignoré (le télécharger avec la cellule de téléchargement NOAA-15).')

### Nettoyage de la table horaire (utilisée dans toute la suite)
Une ligne de la table = une heure × une tranche de L, avec `p6_sum` (somme des flux des mesures de 2 s), `n`
(nombre de mesures), `p6_max` (plus grande mesure) et `omni_p1_*` (canal protons > 25 MeV du même télescope).
Le flux moyen d'une case est `p6_sum / n`.

1. **Trous** : une heure sans mesure (`n = 0`) n'a pas de flux. En revanche un flux nul (`p6_sum = 0`) est une
   **vraie mesure** (0 particule comptée) : on le garde.
2. **Pics aberrants** : quelques mesures de 2 s isolées montent jusqu'à ~10⁸ (la même valeur 97 904 272 revient
   plusieurs fois : c'est un artefact de l'instrument). Le 99,99ᵉ percentile des mesures vaut ~10⁶ : au-dessus de
   **10⁷**, la mesure est physiquement impossible. On la retire de sa case horaire (somme − max, nombre − 1).
3. **Protons solaires (SEP)** : P6 est nominalement un canal *protons* (> 6,9 MeV). Dans les ceintures, il compte
   surtout des électrons ≳ 1 MeV, **sauf pendant une éruption solaire à protons** : il mesure alors des protons et
   plus des électrons (c'est le « quand il ne mesure pas des électrons » de l'énoncé). On repère ces heures dans la
   **calotte polaire** (L ≥ 10) : il n'y a pas d'électrons piégés, mais les protons solaires y entrent directement
   par les lignes de champ ouvertes. Une heure est contaminée si le P6 **ou** le canal protons > 25 MeV de la
   calotte dépasse **médiane + 10 × MAD** (MAD = écart absolu médian × 1,4826, une mesure de dispersion robuste)
   **pendant au moins 2 heures consécutives** (une éruption dure des heures ou des jours ; une heure isolée est un
   simple pic). Le masque est élargi de ±1 h, et les heures marquées sont **retirées pour toutes les tranches de L**.

   *Pourquoi pas le canal 25 MeV dans la ceinture elle-même (première version du code) ?* Il réagit aussi aux
   électrons très intenses : début mai 2023, il dépassait son seuil alors que la calotte polaire restait au niveau
   du bruit (pas de protons solaires). Cette règle aurait retiré de vrais événements de fort flux d'électrons.
4. **OMNI et synchronisation** : les deux tables sont sur la même grille horaire UTC (début d'heure). On vérifie
   qu'il n'y a ni doublon ni date hors grille ; les valeurs de remplissage OMNI sont déjà passées à NaN lors de leur récupération.

In [ ]:
rapport_noaa = {'lignes au départ (heure x tranche de L)': len(noaa)}

# 1. Trous : une heure sans mesure n'a pas de flux (un flux nul, lui, est une vraie mesure)
noaa = noaa[noaa.n > 0].copy()
rapport_noaa['trous retirés (n = 0)'] = rapport_noaa['lignes au départ (heure x tranche de L)'] - len(noaa)

# 2. Pics aberrants : mesure de 2 s > PIC_MAX retirée de sa case horaire
PIC_MAX = 1e7
pic = noaa.p6_max > PIC_MAX
noaa.loc[pic, 'p6_sum'] -= noaa.loc[pic, 'p6_max']
noaa.loc[pic, 'n'] -= 1
noaa = noaa[noaa.n > 0]
rapport_noaa['cases avec un pic > 1e7 corrigées'] = int(pic.sum())

# 3. Protons solaires : P6 ou protons > 25 MeV de la calotte polaire > médiane + 10 MAD pendant >= 2 h,
#    masque élargi de +-1 h
SEP_K = 10
calotte = noaa[noaa.Lbin == L_POLAR].set_index('time')

def anormal(s, k=SEP_K):
    """True quand s dépasse médiane + k x MAD (MAD = écart absolu médian x 1,4826)."""
    med = s.median()
    return s > med + k * 1.4826 * (s - med).abs().median()

p6_calotte = calotte.p6_sum / calotte.n
p1_calotte = calotte.omni_p1_sum / calotte.omni_p1_n.where(calotte.omni_p1_n > 0)
sep = anormal(p6_calotte) | anormal(p1_calotte.dropna()).reindex(calotte.index, fill_value=False)
sep = sep.reindex(pd.date_range(START, END, freq='h', inclusive='left'), fill_value=False)   # grille horaire complète
sep = sep & (sep.shift(1, fill_value=False) | sep.shift(-1, fill_value=False))   # au moins 2 heures consécutives
h = sep.index[sep]
heures_sep = h.union(h - pd.Timedelta('1h')).union(h + pd.Timedelta('1h'))
avant = len(noaa)
noaa = noaa[~noaa.time.isin(heures_sep)]
rapport_noaa['heures contaminées par des protons'] = len(heures_sep)
rapport_noaa['lignes retirées (protons)'] = avant - len(noaa)
rapport_noaa['lignes gardées'] = len(noaa)

# 4. OMNI et synchronisation : même grille horaire, pas de doublon
assert omni.index.is_unique and omni.index.is_monotonic_increasing
assert (noaa.time == noaa.time.dt.floor('h')).all() and (omni.index == omni.index.floor('h')).all()
rapport_noaa['heures NOAA-15 absentes de la grille OMNI'] = int((~noaa.time.isin(omni.index)).sum())

for k, v in rapport_noaa.items():
    print(f'{k:<45}{v:>10}')
jours_sep = pd.Series(heures_sep.floor('D')).value_counts().sort_index()
print('Jours avec au moins 3 heures contaminées :', ', '.join(f'{d:%Y-%m-%d}' for d in jours_sep[jours_sep >= 3].index))

fig, ax = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
for a, s, nom in ((ax[0], p6_calotte, 'P6 calotte'), (ax[1], p1_calotte, 'protons > 25 MeV calotte')):
    a.semilogy(s.index, s.clip(lower=s[s > 0].min()), lw=0.4, color='C0')
    m = s.index.isin(heures_sep)
    a.plot(s.index[m], s[m].clip(lower=s[s > 0].min()), '.', ms=2, color='C3', label='heures retirées')
    a.set_ylabel(nom)
ax[0].set_title('Détection des éruptions solaires à protons dans la calotte polaire (L ≥ 10)')
ax[0].legend(fontsize=8)
fig.tight_layout()
savefig(fig, '2_protons_solaires.png')
plt.show()

## Choix de la région des ceintures de radiation (plage de L)

**Le paramètre L** : distance (en rayons terrestres) à laquelle la ligne de champ magnétique sur laquelle se trouve
le satellite croise l'équateur magnétique. NOAA-15 est en orbite basse (~810 km) mais, en passant d'un pôle à
l'autre, il traverse toutes les lignes de champ, donc toutes les valeurs de L. Le flux mesuré dépend énormément de L :
il faut choisir **quelle région** on cherche à prédire.

**Ce que dit la physique :**
* **L < 2 : ceinture interne.** Peuplée de protons piégés très énergétiques (Anomalie Sud-Atlantique) : P6 y compte
  surtout des **protons**, pas des électrons → à exclure.
* **L ≈ 2–3 : zone « slot ».** Quasi vide d'électrons, sauf après de très forts orages.
* **L ≈ 3–7 : ceinture externe.** C'est là que se trouvent les électrons relativistes (~1 MeV) qui varient fortement
  avec le vent solaire : c'est la région qui nous intéresse.
* **L > 7 :** on sort des ceintures (lignes de champ ouvertes ou très étirées) → flux au niveau du bruit.

**Région retenue : 3 ≤ L < 7**, sur indication de l'enseignant (une plage plus étroite ferait perdre des données).
Elle couvre toute la ceinture externe. **Critères de vérification**, sur les données 2020–2024 ci-dessous. La région doit :
1. avoir un flux **nettement au-dessus du bruit de fond** de l'instrument (sinon on prédit du bruit) ;
2. être **variable** dans le temps (sinon il n'y a rien à prédire) ;
3. être **liée au vent solaire** (sinon les paramètres OMNI ne serviront à rien pour prédire).

### Carte L – temps du flux
Pour chaque jour et chaque tranche de L, flux moyen = Σ flux / Σ mesures. L'échelle de couleur est logarithmique
(le flux varie sur plusieurs ordres de grandeur). L'indice Dst est tracé en dessous : les fortes valeurs négatives
signalent les orages géomagnétiques.

**Pixels isolés aberrants :** quelques cases (1 jour × 1 tranche de L) sont plus de 3 fois (0,5 en log10) au-dessus
ou en dessous **à la fois du jour d'avant et du jour d'après**, sans que les tranches de L voisines bougent. Ce n'est
pas de la physique : un vrai orage dure plusieurs jours et touche plusieurs tranches de L à la fois. Ce sont des
restes de mesures parasites (une poignée de mesures de 2 s anormales dans une case peu remplie). Pour la carte, ces
cases sont remplacées par la **moyenne du jour d'avant et du jour d'après** (même tranche de L).

**Trous de la carte :** les jours sans mesure (fichiers absents chez NOAA, éruptions à protons retirées) sont interpolés
dans le temps, tranche de L par tranche de L, s'ils durent au plus 5 jours. Le long trou d'octobre 2023 reste en gris.

In [ ]:
# Niveau de bruit : dans la calotte polaire (L >= 10) il n'y a pas d'électrons piégés,
# P6 n'y mesure que le bruit de fond de l'instrument
pc = noaa[noaa.Lbin == L_POLAR]
NOISE = (pc.p6_sum / pc.n).median()
print(f'Bruit de fond P6 (médiane horaire en calotte polaire) : {NOISE:.0f} #/cm²/s/sr')

# Moyenne journalière par tranche de L (L < 8)
belt = noaa[noaa.Lbin < L_POLAR]
d = belt.groupby([belt.time.dt.floor('D'), 'Lbin'])[['p6_sum', 'n']].sum()
carte = (d.p6_sum / d.n).unstack('Lbin')
carte = carte.reindex(pd.date_range(START, END, freq='D', inclusive='left'))   # jours manquants -> NaN

# Pixels isolés aberrants -> moyenne du jour d'avant et du jour d'après (même tranche de L)
log_carte = np.log10(carte.clip(lower=NOISE / 2))
avant, apres = log_carte.shift(1), log_carte.shift(-1)
SAUT = 0.5                                                 # facteur 10^0,5 ≈ 3

def isole_en_L(m):
    """Garde les pixels marqués dont aucune tranche de L voisine n'est marquée le même jour."""
    return m & ~(m.shift(1, axis=1, fill_value=False) | m.shift(-1, axis=1, fill_value=False))

aberrant = (isole_en_L(log_carte - np.fmax(avant, apres) > SAUT)      # pic isolé vers le haut
            | isole_en_L(np.fmin(avant, apres) - log_carte > SAUT))   # creux isolé vers le bas
voisins = (avant.fillna(apres) + apres.fillna(avant)) / 2
log_carte = log_carte.mask(aberrant, voisins)

# Trous courts de la carte : interpolation dans le temps, tranche de L par tranche de L
TROU_MAX_JOURS = 5

def combler(colonne):
    """Interpole les trous d'au plus TROU_MAX_JOURS ; les trous plus longs restent vides."""
    manque = colonne.isna()
    longueur = manque.groupby((manque != manque.shift()).cumsum()).transform('sum')
    return colonne.interpolate(limit_area='inside').where(~manque | (longueur <= TROU_MAX_JOURS))

vides_avant = int(log_carte.isna().values.sum())
log_carte = log_carte.apply(combler)
jours_vides = log_carte.index[log_carte.isna().all(axis=1)]
print(f'Cases vides de la carte : {vides_avant} -> {int(log_carte.isna().values.sum())} ; '
      f'jours restés vides : {len(jours_vides)} ({jours_vides.min():%d/%m/%Y} -> {jours_vides.max():%d/%m/%Y})')
print(f'Pixels isolés aberrants remplacés : {int(aberrant.values.sum())} sur {int(carte.notna().values.sum())}')

fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('0.85')                                      # jours manquants en gris
im = ax[0].imshow(log_carte.T.values, aspect='auto', origin='lower', cmap=cmap,
                  extent=[mdates.date2num(START), mdates.date2num(END), 1, 8], interpolation='nearest')
for l in (L_LO, L_HI):
    ax[0].axhline(l, color='w', ls='--', lw=1)
ax[0].grid(False)
ax[0].set_ylabel('L (IGRF)')
ax[0].set_title('NOAA-15 P6 (électrons ~1 MeV) : flux moyen journalier par tranche de L, 2020–2024')
fig.colorbar(im, ax=ax, label='log10 flux P6 (#/cm²/s/sr)', pad=0.01, fraction=0.03)
ax[1].plot(omni.index, omni['Dst_nT'], color='C0', lw=0.5)
ax[1].set_ylabel('Dst (nT)')
ax[1].xaxis_date()
savefig(fig, '3_carte_L_temps.png')
plt.show()

### Profil en L : niveau, variabilité et lien avec le vent solaire
On calcule, pour chaque tranche de L, le flux moyen sur des fenêtres de 12 h (la résolution du projet), puis :
* le **flux moyen** (linéaire) → où se trouve le cœur de la ceinture ;
* la **médiane** et les percentiles 5 % / 95 % du log10 du flux → niveau par rapport au bruit (critère 1) ;
* le **% de fenêtres où le flux dépasse 2 × le bruit** (critère 1) ;
* l'**écart-type du log10 du flux** → variabilité (critère 2) ;
* la **corrélation de Spearman** entre le log10 du flux et la vitesse du vent solaire V **24 h plus tôt**
  (le vent solaire met 1 à 2 jours à accélérer les électrons) → lien avec le vent solaire (critère 3).

In [ ]:
d12 = belt.groupby([pd.Grouper(key='time', freq=STEP), 'Lbin'])[['p6_sum', 'n']].sum()
f12 = (d12.p6_sum / d12.n).unstack('Lbin')                # flux 12 h : une colonne par tranche de L
logf = np.log10(f12.clip(lower=NOISE / 2))
V12 = omni['V_kms'].resample(STEP).mean()

profil = pd.DataFrame({
    'flux moyen': f12.mean(),
    'log10 P5': logf.quantile(0.05),
    'log10 médiane': logf.median(),
    'log10 P95': logf.quantile(0.95),
    '% > 2×bruit': (f12 > 2 * NOISE).mean() * 100,
    'écart-type log10': logf.std(),
    'Spearman V(t-24h)': logf.apply(lambda c: c.corr(V12.shift(2).reindex(c.index), method='spearman')),
})
profil.index.name = 'L (bord gauche)'

LBIN_W = 0.25                                             # largeur d'une tranche de L
flux_moy = profil['flux moyen']
print(f'Flux moyen maximal : {flux_moy.max():.0f} #/cm²/s/sr pour {flux_moy.idxmax()} <= L < {flux_moy.idxmax() + LBIN_W}')
print(f'Région retenue : {L_LO} <= L < {L_HI}')
display(profil.round(2))

x = profil.index + LBIN_W / 2                             # centre des tranches
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
ax[0].plot(x, profil['flux moyen'], color='C0', lw=2)
ax[0].set_ylabel('flux moyen (#/cm²/s/sr)')
ax[0].set_title('Flux moyen')
ax[1].fill_between(x, profil['log10 P5'], profil['log10 P95'], color='C0', alpha=0.25, label='P5 – P95')
ax[1].plot(x, profil['log10 médiane'], color='C0', lw=2, label='médiane')
ax[1].axhline(np.log10(NOISE), color='C3', ls='--', label='bruit')
ax[1].set_ylabel('log10 flux (12 h)')
ax[1].set_title('1. Niveau')
ax[1].legend(fontsize=7)
ax[2].plot(x, profil['écart-type log10'], color='C0', lw=2)
ax[2].set_ylabel('écart-type de log10 flux')
ax[2].set_title('2. Variabilité')
ax[3].plot(x, profil['Spearman V(t-24h)'], color='C0', lw=2)
ax[3].set_ylabel('ρ Spearman')
ax[3].set_title('3. Lien avec V 24 h avant')
for a in ax:
    a.axvspan(L_LO, L_HI, color='C1', alpha=0.2)
    a.set_xlabel('L')
fig.suptitle(f'Profil en L (fenêtres de 12 h, 2020–2024) — zone orange : région retenue {L_LO} ≤ L < {L_HI}')
fig.tight_layout()
savefig(fig, '3_profil_L.png')
plt.show()

### Conclusion : vérification de la région retenue

**Ce que montre la carte L – temps :**
* une bande **constante à L ≈ 1,25–1,75** : les protons de la ceinture interne, insensibles aux orages → exclue ;
* une **zone « slot » sombre à L ≈ 2–3**, remplie seulement lors des orages les plus forts (2023, mai 2024 :
  Dst ≈ −400 nT), quand la ceinture externe est poussée vers la Terre ;
* une **ceinture externe à L ≈ 3–7, plus intense vers L ≈ 4–5,** qui s'allume et s'éteint au rythme de l'activité solaire : c'est la cible ;
* au-delà de L ≈ 7, le flux reste au niveau du bruit.

La cellule ci-dessous vérifie que la région retenue remplit les trois critères (flux moyenné sur toute la région,
fenêtres de 12 h) et la compare aux deux régions décalées d'une demi-unité de L, ainsi qu'à la partie la plus intense de la ceinture.

In [ ]:
def criteres(lo, hi):
    """Les 3 critères de choix pour la région lo <= L < hi (flux 12 h moyenné sur toute la région)."""
    g = belt[(belt.Lbin >= lo) & (belt.Lbin < hi)].groupby(pd.Grouper(key='time', freq=STEP))[['p6_sum', 'n']].sum()
    f = (g.p6_sum / g.n).dropna()
    lf = np.log10(f.clip(lower=NOISE / 2))
    return {'flux moyen': f.mean(),
            '% > 2×bruit': (f > 2 * NOISE).mean() * 100,
            'écart-type log10': lf.std(),
            'Spearman V(t-24h)': lf.corr(V12.shift(2).reindex(lf.index), method='spearman')}

regions = {f'{L_LO} ≤ L < {L_HI} (retenue)': criteres(L_LO, L_HI),
           f'{L_LO - 0.5} ≤ L < {L_HI - 0.5}': criteres(L_LO - 0.5, L_HI - 0.5),
           f'{L_LO + 0.5} ≤ L < {L_HI + 0.5}': criteres(L_LO + 0.5, L_HI + 0.5),
           '3.5 ≤ L < 5.25 (cœur de la ceinture)': criteres(3.5, 5.25)}
display(pd.DataFrame(regions).T.round(2))

**Lecture (2020–2024) : la région 3 ≤ L < 7** contient le maximum du flux moyen (L ≈ 4,25) et toute la ceinture externe.
Moyennée sur toute la région, elle remplit les trois critères :
1. flux **au-dessus de 2 × le bruit dans 92 % des fenêtres** de 12 h ;
2. **variabilité** forte (écart-type de log10 flux ≈ 0,51) : il y a quelque chose à prédire ;
3. **lien avec le vent solaire** (Spearman avec V 24 h avant ≈ 0,36), au moins aussi bon que les autres plages testées.

Par rapport à la partie la plus intense de la ceinture (3,5 ≤ L < 5,25), elle contient **deux fois plus de mesures**
par fenêtre (médiane ≈ 3 250 contre ≈ 1 570), pour un flux moyen plus faible (2 900 contre 4 700) et une variabilité un
peu moindre (0,51 contre 0,56). Décaler la plage d'une demi-unité change peu les critères.

Limites : prises une par une, les tranches du bord sont souvent proches du bruit (au-dessus de 2 × le bruit dans
23 à 44 % des fenêtres pour L 3–3,5, et 19 à 44 % pour L 6,25–7). La ceinture se déplace aussi (vers la Terre pendant
les forts orages), et une plage fixe ne la suit pas toujours ; comparer plusieurs plages de L est d'ailleurs proposé en
bonus par l'énoncé.

## Moyennes sur 12 h

Toutes les données sont ramenées sur des **fenêtres de 12 h** (00:00–12:00 et 12:00–24:00 UTC), la résolution des
prédictions du Step 2 :
* **flux** : on garde les tranches de la région retenue (`L_LO ≤ L < L_HI`, choisie plus haut) et on calcule
  flux 12 h = Σ `p6_sum` / Σ `n`, c'est-à-dire la moyenne du **flux linéaire** de toutes les mesures de la
  fenêtre (puis son log10). Une fenêtre est déclarée **invalide** (NaN) si elle contient moins de 25 % du nombre
  médian de mesures : trop peu de passages dans la région pour que la moyenne soit fiable ;
* **trous du flux** : les trous courts (jusqu'à 5 jours) sont **interpolés** : le log10 du flux varie en ligne droite
  entre la dernière fenêtre mesurée avant le trou et la première après. Le long trou d'octobre 2023 (13 jours) reste
  vide : une droite sur une telle durée ne représenterait plus rien. La colonne `interpole` indique ces fenêtres ;
* `couverture` = nombre d'heures de la fenêtre où NOAA-15 est passé dans la région / 12 ;
* **paramètres OMNI** : moyenne sur la fenêtre, NaN s'il y a moins de 6 heures valides sur 12.

La table est enregistrée dans `data/table_12h.parquet` : c'est l'**entrée du Step 2**.

In [ ]:
fen = pd.date_range(START, END, freq=STEP, inclusive='left')   # début de chaque fenêtre de 12 h

# Flux de la région : somme des flux / nombre de mesures sur chaque fenêtre
reg = noaa[(noaa.Lbin >= L_LO) & (noaa.Lbin < L_HI)]
g = reg.groupby(pd.Grouper(key='time', freq=STEP))[['p6_sum', 'n']].sum().reindex(fen, fill_value=0)
heures = reg.groupby(pd.Grouper(key='time', freq=STEP)).time.nunique().reindex(fen, fill_value=0)
N_MIN = 0.25 * g.n[g.n > 0].median()
flux12 = (g.p6_sum / g.n).where(g.n >= N_MIN)

# Trous courts du flux : interpolation linéaire du log10 ; les trous plus longs que TROU_MAX restent vides
TROU_MAX = 10                                             # en fenêtres de 12 h, soit 5 jours
mesure = flux12.notna()
longueur_trou = (~mesure).groupby((mesure != mesure.shift()).cumsum()).transform('sum')
log_flux12 = np.log10(flux12).interpolate().where(mesure | (longueur_trou <= TROU_MAX))

# OMNI : moyenne sur 12 h si au moins 6 heures valides
omni12 = omni.resample(STEP).mean().where(omni.resample(STEP).count() >= 6).reindex(fen)

tab = pd.DataFrame({'flux': 10 ** log_flux12, 'log_flux': log_flux12, 'n': g.n, 'couverture': heures / 12,
                    'interpole': ~mesure & log_flux12.notna()})
tab = tab.join(omni12)
tab.index.name = 'time'
tab.to_parquet(os.path.join(DATA, 'table_12h.parquet'))

print(f'Région {L_LO} <= L < {L_HI} : {len(tab)} fenêtres de 12 h, '
      f'{(~mesure).sum()} sans flux mesuré (seuil : {N_MIN:.0f} mesures), dont {tab.interpole.sum()} interpolées '
      f'et {tab.flux.isna().sum()} laissées vides')
print('% manquant par colonne :', tab.isna().mean().mul(100).round(1).to_dict())
tab.head()

## Visualisation des séries temporelles

### Flux d'électrons en fonction du temps

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.semilogy(tab.index, tab.flux, lw=0.7, color='C0')
ax.plot(tab.index[tab.interpole], tab.flux[tab.interpole], '.', ms=3, color='C1', label='fenêtres interpolées')
ax.axhline(NOISE, color='C3', ls='--', lw=1, label='bruit de fond')
ax.set_ylabel('flux P6 (#/cm²/s/sr)')
ax.set_title(f'Flux d\'électrons ~1 MeV moyenné sur 12 h, région {L_LO} ≤ L < {L_HI}')
ax.legend(fontsize=8)
savefig(fig, '5_flux_temps.png')
plt.show()

### L en fonction du temps
En haut : position de NOAA-15 en L heure par heure sur deux jours (couleur = nombre de mesures de 2 s dans chaque
case heure × tranche de L, échelle log). En ~25 min, le satellite va de l'équateur (L ≈ 1) au pôle (L > 10) : il
traverse donc **toutes les valeurs de L plusieurs fois par heure**, la région retenue (lignes rouges) comprise. Il
passe plus de temps à bas L (près de l'équateur, les lignes de champ défilent lentement). En bas : nombre de mesures
dans la région par fenêtre de 12 h, sur les 5 ans (les creux = jours sans données ou heures retirées).

In [ ]:
from matplotlib.colors import LogNorm

J0 = pd.Timestamp('2022-03-10')
deux_jours = noaa[(noaa.time >= J0) & (noaa.time < J0 + pd.Timedelta('2D')) & (noaa.Lbin < L_POLAR)]
pos = deux_jours.pivot_table(index='Lbin', columns='time', values='n', aggfunc='sum')

fig, ax = plt.subplots(2, 1, figsize=(11, 6), gridspec_kw={'height_ratios': [1.3, 1]})
im = ax[0].pcolormesh(mdates.date2num(pos.columns), pos.index, pos.values, cmap='viridis', shading='nearest',
                     norm=LogNorm())
fig.colorbar(im, ax=ax[0], label='mesures / heure', pad=0.01)
for l in (L_LO, L_HI):
    ax[0].axhline(l, color='C3', ls='--', lw=1.2)
ax[0].xaxis_date()
ax[0].grid(False)
ax[0].set_ylabel('L')
ax[0].set_title(f'Passages de NOAA-15 en L, {J0:%d/%m/%Y} + 2 jours ')
ax[1].plot(tab.index, tab.n, lw=0.5, color='C0')
ax[1].axhline(N_MIN, color='C3', ls='--', lw=1, label='minimum pour une fenêtre valide')
ax[1].set_ylabel('mesures / 12 h')
ax[1].set_title(f'Nombre de mesures dans la région {L_LO} ≤ L < {L_HI} par fenêtre de 12 h')
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, '5_L_temps.png')
plt.show()

### Paramètres solaires en fonction du temps
Le flux (en haut) est tracé avec les paramètres solaires et géomagnétiques pour les comparer à l'œil : vitesse du
vent solaire V, composante Bz du champ magnétique interplanétaire, flux radio solaire F10.7 (activité du Soleil,
qui augmente sur 2020–2024 : montée du cycle 25), indices Kp et Dst (orages géomagnétiques : Dst très négatif).

In [ ]:
series = [('log_flux', 'log10 flux'), ('V_kms', 'V (km/s)'), ('Bz_nT', 'Bz (nT)'),
          ('F107_sfu', 'F10.7 (sfu)'), ('Kp', 'Kp'), ('Dst_nT', 'Dst (nT)')]
fig, ax = plt.subplots(len(series), 1, figsize=(11, 10), sharex=True)
for a, (col, nom) in zip(ax, series):
    a.plot(tab.index, tab[col], lw=0.5, color='C1' if col == 'log_flux' else 'C0')
    a.set_ylabel(nom)
ax[0].set_title('Flux et paramètres solaires / géomagnétiques, moyennes sur 12 h')
fig.tight_layout()
savefig(fig, '5_parametres_solaires.png')
plt.show()

## Statistiques descriptives, relations, outliers

### Statistiques descriptives

In [ ]:
display(tab.describe().T)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(tab.log_flux.dropna(), bins=50, color='C0')
ax.set_xlabel('log10 flux (12 h)')
ax.set_ylabel('nombre de fenêtres')
ax.set_title('Distribution du flux')
savefig(fig, '6_histogramme_flux.png')
plt.show()

### Relations entre le flux et les paramètres solaires
Corrélation de **Spearman** (sur les rangs : elle mesure une relation monotone, même non linéaire, et elle est peu
sensible aux valeurs extrêmes), entre toutes les colonnes au même instant t.

In [ ]:
COLS = ['log_flux', 'V_kms', 'Bz_nT', 'F107_sfu', 'N_cm3', 'P_nPa', 'E_mVm', 'B_nT', 'Kp', 'Dst_nT', 'AE_nT']
corr = tab[COLS].corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(COLS)))
ax.set_xticklabels(COLS, rotation=45, ha='right')
ax.set_yticks(range(len(COLS)))
ax.set_yticklabels(COLS)
ax.grid(False)
for i in range(len(COLS)):
    for j in range(len(COLS)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, label='corrélation de Spearman')
ax.set_title('Corrélations au même instant (fenêtres de 12 h)')
fig.tight_layout()
savefig(fig, '6_correlations.png')
plt.show()

### Corrélations retardées
Le flux d'électrons réagit au vent solaire **avec retard** (les électrons sont accélérés en 1 à 2 jours). On calcule
donc la corrélation entre le flux à t et chaque paramètre à **t − k × 12 h** (k = 0 à 8, soit jusqu'à 4 jours
avant). C'est ce qui indique quels paramètres seront utiles pour prédire le flux à +12 → +48 h.

In [ ]:
LAGS = range(0, 9)
lag = pd.DataFrame({c: [tab.log_flux.corr(tab[c].shift(k), method='spearman') for k in LAGS]
                    for c in COLS if c != 'log_flux'}, index=[12 * k for k in LAGS])
lag.index.name = 'retard (h)'

fig, ax = plt.subplots(figsize=(9, 4))
for c in lag.columns:
    ax.plot(lag.index, lag[c], 'o-', ms=3, label=c)
ax.axhline(0, color='k', lw=0.5)
ax.set_xticks(lag.index)
ax.set_xlabel('paramètre pris k × 12 h avant le flux (h)')
ax.set_ylabel('corrélation de Spearman avec log10 flux')
ax.set_title('Corrélations retardées : flux(t) vs paramètre(t − retard)')
ax.legend(fontsize=7, ncol=2)
savefig(fig, '6_correlations_retardees.png')
plt.show()

meilleur = pd.DataFrame({'retard le plus corrélé (h)': lag.abs().idxmax(),
                         'corrélation': [lag.loc[lag[c].abs().idxmax(), c] for c in lag.columns]})
display(meilleur.sort_values('corrélation', key=abs, ascending=False).round(2))

**Lecture :**
* la **vitesse du vent solaire V** est le meilleur indicateur, avec un **retard d'environ 2,5 jours** (ρ ≈ 0,45) :
  un vent rapide aujourd'hui annonce un flux élevé 2–3 jours plus tard (temps d'accélération des électrons) ;
* **Kp, AE** (activité géomagnétique, ρ ≈ 0,4 à ~3,5 jours) et **Dst** (ρ ≈ −0,38 : un orage, Dst < 0, précède une
  hausse du flux) vont dans le même sens ;
* la **densité N** est anti-corrélée (ρ ≈ −0,30 à 24 h) : un vent dense et lent va avec un flux plus faible ;
* **Bz** est peu corrélé en moyenne 12 h : son signe change très vite, la moyenne efface son effet.

→ Pour le Step 2, les valeurs passées de V, Kp, AE, Dst et N sur les 1 à 4 derniers jours sont les entrées les plus
utiles.

### Valeurs extrêmes (outliers)
Règle de l'écart interquartile (IQR) appliquée à chaque colonne de la table 12 h : une valeur est extrême si elle
sort de [Q1 − 1,5·IQR ; Q3 + 1,5·IQR]. On regarde ensuite **quand** elles se produisent : si elles tombent pendant
des orages géomagnétiques (Dst très négatif), ce sont de **vrais phénomènes physiques**, pas des erreurs.

In [ ]:
q1, q3 = tab[COLS].quantile(0.25), tab[COLS].quantile(0.75)
iqr = q3 - q1
extreme = (tab[COLS] < q1 - 1.5 * iqr) | (tab[COLS] > q3 + 1.5 * iqr)
display(pd.DataFrame({'fenêtres extrêmes': extreme.sum(), '%': (extreme.mean() * 100).round(1)}).T)

# Les fenêtres les plus extrêmes du vent solaire et du Dst : à quelles dates ?
print('5 fenêtres de Dst le plus négatif :')
display(tab.nsmallest(5, 'Dst_nT')[['log_flux', 'interpole', 'V_kms', 'Bz_nT', 'Kp', 'Dst_nT']].round(2))
orage = tab.Dst_nT < -50
print(f'Part des fenêtres extrêmes en V, Bz ou Kp tombant pendant un orage (Dst < -50 nT) : '
      f'{orage[extreme[["V_kms", "Bz_nT", "Kp"]].any(axis=1)].mean():.0%} '
      f'(contre {orage.mean():.0%} de toutes les fenêtres)')

**Lecture :**
* le **log10 du flux n'a aucune valeur extrême** : en échelle log, sa distribution est compacte (le flux varie sur
  2,6 ordres de grandeur, de ~10² à ~4·10⁴, sans valeur isolée) ;
* les valeurs extrêmes des paramètres solaires (V, Bz, Kp…) tombent **7 fois plus souvent pendant un orage**
  (23 % contre 3 %) : ce sont de **vrais phénomènes physiques** (tempête de mai 2024 : Dst = −349 nT, Kp = 8,7 ;
  octobre 2024 : Dst = −270 nT). On les **garde** : ce sont justement les situations qu'on veut savoir prédire ;
* pendant la tempête de mai 2024, il n'y a pas de flux mesuré exploitable : une éruption à protons solaires a lieu en
  même temps et ces heures ont été retirées lors du nettoyage (P6 ne mesurait plus des électrons). Les valeurs
  affichées pour ces fenêtres sont interpolées.

## Seuil du 95ᵉ percentile : définition des « événements »
Un **événement** est une fenêtre de 12 h où le flux dépasse le **95ᵉ percentile** de tous les flux 12 h : par
construction, 5 % des fenêtres. Des fenêtres consécutives au-dessus du seuil forment un même épisode.

Le seuil et les événements sont calculés sur les **fenêtres réellement mesurées** (pas sur les fenêtres interpolées).

⚠️ Pour le Step 2, ce seuil sera **recalculé sur la seule période d'entraînement** : le calculer sur toute la période
utiliserait de l'information sur le futur (le test).

In [ ]:
flux_mesure = tab.flux[~tab.interpole].dropna()
SEUIL_P95 = np.percentile(flux_mesure, P_EVENT)
evenement = (tab.flux > SEUIL_P95) & ~tab.interpole
episode = (evenement & ~evenement.shift(fill_value=False)).cumsum()[evenement]   # numéro d'épisode
episodes = pd.DataFrame({'début': episode.index.to_series().groupby(episode).min(),
                         'fenêtres de 12 h': episode.groupby(episode).size(),
                         'flux max': tab.flux[evenement].groupby(episode).max().round(0)})

print(f'Seuil P{P_EVENT} = {SEUIL_P95:.0f} #/cm²/s/sr (log10 = {np.log10(SEUIL_P95):.2f})')
print(f'{evenement.sum()} fenêtres au-dessus du seuil ({evenement.sum() / len(flux_mesure):.1%}), '
      f'regroupées en {len(episodes)} épisodes')
print('Épisodes par année :', episodes['début'].dt.year.value_counts().sort_index().to_dict())

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.semilogy(tab.index, tab.flux, lw=0.6, color='C0')
ax.axhline(SEUIL_P95, color='C3', ls='--', label=f'seuil P{P_EVENT} = {SEUIL_P95:.0f}')
ax.plot(tab.index[evenement], tab.flux[evenement], '.', ms=4, color='C3', label='événements')
ax.set_ylabel('flux P6 (#/cm²/s/sr)')
ax.set_title(f'Événements de fort flux (> P{P_EVENT}), région {L_LO} ≤ L < {L_HI}')
ax.legend(fontsize=8)
savefig(fig, '7_seuil_P95.png')
plt.show()
episodes.sort_values('flux max', ascending=False).head(10)

**Lecture :** seuil P95 = **≈ 10 600 #/cm²/s/sr** (log10 ≈ 4,02), soit **178 fenêtres de 12 h regroupées en
58 épisodes**. Les épisodes les plus longs durent plusieurs jours (avril–mai 2023 : 23 fenêtres, soit 11,5 jours ;
septembre 2022 : 17 fenêtres, soit 8,5 jours). 12 épisodes tombent en 2024, l'année de test du Step 2. Les événements
restent rares (5 % des fenêtres), d'où l'importance de métriques adaptées aux classes déséquilibrées (F1, AUC).

# 4. Step 2 : modélisation par apprentissage automatique

## 4.1 Régression : prédire le flux à +12, +24, +36 et +48 h

### Données d'entrée et découpage entraînement / test
On part de la **table 12 h** produite plus haut (`data/table_12h.parquet`).

**Ce que le modèle reçoit (les entrées X) :** à un instant t, on ne connaît que le passé. On donne donc au modèle
le log10 du flux et les paramètres solaires **à t, t − 12 h et t − 24 h** (le flux réagit au vent solaire avec 1 à
2 jours de retard, les valeurs passées sont donc utiles).

**Ce qu'il doit prédire (les cibles Y) :** le log10 du flux **12, 24, 36 et 48 h plus tard**. On prédit le log10
et pas le flux lui-même, car le flux varie sur plusieurs ordres de grandeur : sans log, les quelques valeurs
d'orage écraseraient tout le reste.

**Découpage honnête (l'énoncé insiste) :** le test doit jouer le rôle d'un **futur que le modèle n'a jamais vu**.
On coupe donc **dans le temps**, et pas au hasard :
* **entraînement : 2020 → 2023** (le modèle apprend) ;
* **test : 2024** (on mesure ses performances), qui contient en plus le gros orage de mai 2024.

Un découpage au hasard tricherait : deux fenêtres de 12 h voisines se ressemblent beaucoup, le modèle aurait vu
« la veille » de presque chaque date de test. On retire aussi les 2 derniers jours de 2023 de l'entraînement :
leurs cibles à +48 h tombent déjà en 2024.

**Fenêtres interpolées :** boucher un trou utilise la mesure d'après le trou, c'est-à-dire le futur. Le modèle peut
apprendre dessus, mais une fenêtre du test n'est gardée que si ses **entrées et ses cibles sont toutes de vraies
mesures** : sinon il verrait une partie de la réponse.

In [ ]:
tab = pd.read_parquet(os.path.join(DATA, 'table_12h.parquet'))

PARAMS = ['log_flux', 'V_kms', 'Bz_nT', 'F107_sfu', 'N_cm3', 'P_nPa', 'E_mVm', 'B_nT', 'Kp', 'Dst_nT', 'AE_nT']
HORIZONS = [1, 2, 3, 4]                 # en pas de 12 h : +12, +24, +36, +48 h
TEST_START = pd.Timestamp('2024-01-01')

# Entrées X(t) : chaque paramètre à t, t-12 h et t-24 h (shift(k) = valeur k pas de 12 h plus tôt)
X = pd.DataFrame({f'{c}_t-{12 * k}h': tab[c].shift(k) for c in PARAMS for k in (0, 1, 2)})
# Cibles Y(t) : log10 du flux 12, 24, 36, 48 h plus tard (shift(-h) = valeur h pas plus tard)
Y = pd.DataFrame({f'+{12 * h}h': tab['log_flux'].shift(-h) for h in HORIZONS})

# On ne garde que les instants où toutes les entrées et toutes les cibles existent
ok = X.notna().all(axis=1) & Y.notna().all(axis=1)
train = ok & (X.index < TEST_START - pd.Timedelta(hours=12 * max(HORIZONS)))   # cibles avant 2024
# Test : uniquement des fenêtres dont les entrées (t, t-12 h, t-24 h) et les cibles sont de vraies mesures
interpole = tab['interpole'].astype(bool)
sans_interpolation = ~pd.concat([interpole.shift(k, fill_value=False) for k in [2, 1, 0] + [-h for h in HORIZONS]],
                                axis=1).any(axis=1)
test = ok & (X.index >= TEST_START) & sans_interpolation
X_train, Y_train = X[train], Y[train]
X_test, Y_test = X[test], Y[test]

print(f'{X.shape[1]} entrées : {len(PARAMS)} paramètres x 3 instants (t, t-12 h, t-24 h)')
print(f'Entraînement : {len(X_train)} fenêtres ({X_train.index.min():%Y-%m-%d} -> {X_train.index.max():%Y-%m-%d})')
print(f'Test         : {len(X_test)} fenêtres ({X_test.index.min():%Y-%m-%d} -> {X_test.index.max():%Y-%m-%d})')
print(f'Fenêtres écartées (valeur manquante, cible à cheval sur 2024, ou interpolation dans le test) : {(~(train | test)).sum()}')

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(tab.index, tab['log_flux'], color='0.6', lw=0.6)
ax.plot(X_train.index, tab.loc[X_train.index, 'log_flux'], '.', ms=1.5, color='C0', label='entraînement')
ax.plot(X_test.index, tab.loc[X_test.index, 'log_flux'], '.', ms=1.5, color='C1', label='test')
ax.set_ylabel('log10 flux (12 h)')
ax.set_title(f'Découpage dans le temps : entraînement 2020–2023, test 2024')
ax.legend(markerscale=6)
savefig(fig, '8_decoupage.png')
plt.show()

### Baseline (persistance) et régression linéaire

**La baseline** est le modèle le plus bête possible, qui sert de point de comparaison (l'énoncé l'exige) : la
**persistance**, « le flux dans 12, 24, 36 ou 48 h sera le même que maintenant ». Elle est difficile à battre à
court terme, car le flux change lentement. Un modèle n'a d'intérêt que s'il fait **mieux** qu'elle.

**La régression linéaire** cherche la meilleure combinaison linéaire des 33 entrées :
log10 flux(t + h) ≈ a₀ + a₁·entrée₁ + a₂·entrée₂ + … Un modèle séparé est entraîné pour chaque horizon h.
Les entrées sont d'abord **standardisées** (moyenne 0, écart-type 1, calculés sur l'entraînement seulement) pour
que les coefficients soient comparables entre eux.

**Les métriques** (calculées sur le test 2024, en unités de log10 flux) :
* **RMSE** (racine de l'erreur quadratique moyenne) : erreur typique, pénalise fort les grosses erreurs.
  RMSE = 0,3 signifie une erreur typique d'un facteur 10^0,3 ≈ 2 sur le flux ;
* **MAE** (erreur absolue moyenne) : erreur moyenne, moins sensible aux grosses erreurs ;
* **R²** : part de la variabilité du flux expliquée par le modèle (1 = parfait, 0 = pas mieux que prédire la
  moyenne, négatif = pire que la moyenne).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def scores(y_vrai, y_pred):
    """RMSE, MAE et R² (en unités de log10 flux)."""
    return {'RMSE': np.sqrt(mean_squared_error(y_vrai, y_pred)),
            'MAE': mean_absolute_error(y_vrai, y_pred),
            'R²': r2_score(y_vrai, y_pred)}

predictions = {}   # predictions[modèle] = DataFrame des prédictions sur le test (une colonne par horizon)
resultats = {}     # resultats[modèle] = DataFrame des métriques (une ligne par horizon)

# Baseline : persistance (flux futur = flux actuel)
predictions['Persistance'] = pd.DataFrame({h: X_test['log_flux_t-0h'] for h in Y.columns})

# Régression linéaire : un modèle par horizon (standardisation apprise sur l'entraînement seulement)
predictions['Régression linéaire'] = pd.DataFrame(index=X_test.index)
modeles_lin = {}
for h in Y.columns:
    modele = make_pipeline(StandardScaler(), LinearRegression())
    modele.fit(X_train, Y_train[h])
    modeles_lin[h] = modele
    predictions['Régression linéaire'][h] = modele.predict(X_test)

for nom, pred in predictions.items():
    resultats[nom] = pd.DataFrame({h: scores(Y_test[h], pred[h]) for h in Y.columns}).T

display(pd.concat(resultats, axis=1).round(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for nom, res in resultats.items():
    for a, m in zip(ax, ['RMSE', 'MAE', 'R²']):
        a.plot(res.index, res[m], 'o-', label=nom)
        a.set_title(m)
        a.set_xlabel('horizon')
ax[0].legend(fontsize=8)
fig.suptitle('Erreurs sur le test (2024) selon l\'horizon de prédiction')
fig.tight_layout()
savefig(fig, '8_metriques_lineaire.png')
plt.show()

**Lecture :**
* la **régression linéaire bat la persistance à tous les horizons** : RMSE 0,15 → 0,22 contre 0,20 → 0,28, et
  R² 0,92 → 0,84 contre 0,88 → 0,74. Le gain grandit avec l'horizon (à +48 h, l'erreur baisse de ~22 %) : plus on
  prédit loin, plus le vent solaire des jours précédents apporte d'information que le flux actuel n'a pas ;
* l'erreur augmente avec l'horizon, comme attendu : RMSE = 0,22 à +48 h correspond à une erreur typique d'un
  facteur 10^0,22 ≈ 1,7 sur le flux ;
* **curiosité : la persistance est moins bonne à +12 h qu'à +24 h** (MAE 0,154 contre 0,146). Le flux de la
  fenêtre 12h–24h UTC est **systématiquement plus bas** que celui de la fenêtre 00h–12h (de 0,12 en log10, soit un
  facteur 1,3, dans 83 % des cas, et de 0,10 à 0,14 chaque année). « Même fenêtre hier » (+24 h) ressemble donc plus
  à aujourd'hui que « la fenêtre suivante » (+12 h).
  *Piste non vérifiée :* selon l'heure UTC, NOAA-15 ne survole pas les mêmes longitudes, et à même L le flux mesuré
  à 810 km dépend de la longitude (le champ magnétique n'est pas symétrique). La table horaire ne contient pas la
  longitude : il faudrait repartir des fichiers bruts pour le confirmer.

### Random Forest
Une **forêt aléatoire** (Random Forest) est un ensemble de **arbres de décision**. Chaque arbre pose une suite de
questions simples sur les entrées (« V il y a 24 h > 500 km/s ? », « flux actuel > 10³ ? »…) et arrive à une
prédiction. Chaque arbre est entraîné sur un tirage au hasard des données ; la forêt fait la **moyenne** de tous
les arbres, ce qui rend la prédiction plus stable qu'un arbre seul.

Différence avec la régression linéaire : la forêt peut capter des relations **non linéaires** et des
**combinaisons** d'entrées (par exemple : un vent rapide n'accélère les électrons que si Bz est négatif).

Réglages (fixés à l'avance, sans regarder le test) : 300 arbres ; au moins 5 fenêtres par feuille, pour éviter que
les arbres apprennent par cœur les données d'entraînement ; à chaque question, un tiers des entrées tirées au
hasard. Un modèle par horizon, comme pour la régression linéaire.

L'**importance des entrées** mesure combien chaque entrée a servi à réduire l'erreur dans les arbres : elle dit
sur quoi le modèle s'appuie.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

predictions['Random Forest'] = pd.DataFrame(index=X_test.index)
modeles_rf = {}
for h in Y.columns:
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=1 / 3,
                               random_state=0, n_jobs=-1)
    rf.fit(X_train, Y_train[h])
    modeles_rf[h] = rf
    predictions['Random Forest'][h] = rf.predict(X_test)
resultats['Random Forest'] = pd.DataFrame({h: scores(Y_test[h], predictions['Random Forest'][h])
                                           for h in Y.columns}).T

display(pd.concat(resultats, axis=1).round(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for nom, res in resultats.items():
    for a, m in zip(ax, ['RMSE', 'MAE', 'R²']):
        a.plot(res.index, res[m], 'o-', label=nom)
        a.set_title(m)
        a.set_xlabel('horizon')
ax[0].legend(fontsize=8)
fig.suptitle('Erreurs sur le test (2024) selon l\'horizon de prédiction')
fig.tight_layout()
savefig(fig, '8_metriques_3_modeles.png')
plt.show()

# Importance des entrées (moyenne sur les 4 horizons), les 15 plus utiles
importance = pd.DataFrame({h: modeles_rf[h].feature_importances_ for h in Y.columns}, index=X.columns)
top = importance.mean(axis=1).sort_values().tail(15)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top.index, top.values, color='C2')
ax.set_xlabel('importance moyenne (4 horizons)')
ax.set_title('Random Forest : les 15 entrées les plus utilisées')
fig.tight_layout()
savefig(fig, '8_importance_rf.png')
plt.show()
display(importance.loc[top.index[::-1]].round(3))

**Lecture :**
* la Random Forest **bat la persistance**, mais **fait moins bien que la régression linéaire**, et l'écart grandit
  avec l'horizon (RMSE à +48 h : 0,27 contre 0,22 ; R² : 0,77 contre 0,84). Un modèle plus complexe n'est donc pas
  forcément meilleur : c'est un résultat à discuter dans le rapport ;
* **ce qu'elle utilise** : surtout le flux actuel et passé (t, t − 12 h, t − 24 h), qui pèse de 65 à 83 % de l'importance selon l'horizon.
  Viennent ensuite V, Dst et Kp, dont le poids **augmente avec l'horizon** : à +48 h, le flux actuel en dit
  moins, le vent solaire et l'activité géomagnétique en disent plus (comme vu aux corrélations retardées).

Pourquoi fait-elle moins bien que la régression linéaire ? La question est testée à la fin de la section « Flux prédit et flux observé ».

### Flux prédit et flux observé
Les métriques résument l'erreur en un seul nombre ; ces figures montrent **où** les modèles se trompent (test 2024) :
* **au fil du temps** : le flux observé et les prédictions, à la date visée (t + h). Les coupures sont des fenêtres
  sans vraie mesure ;
* **prédit en fonction d'observé** : un modèle parfait placerait tous les points sur la diagonale ;
* **grandes variations** : les métriques globales sont dominées par les périodes calmes. On recalcule donc la RMSE
  sur les fenêtres où le flux change d'**au moins un facteur 3** (0,5 en log10) entre t et t + h, c'est-à-dire
  les montées et les chutes rapides, les cas qu'on voudrait justement anticiper.

In [ ]:
COULEUR = {'Persistance': 'C0', 'Régression linéaire': 'C1', 'Random Forest': 'C2'}   # mêmes couleurs que les métriques
TRAIT = {'Régression linéaire': '-', 'Random Forest': '--'}
grille = tab.index[tab.index >= TEST_START]               # toutes les fenêtres de 2024, pour garder les coupures

# Au fil du temps : une ligne par horizon
fig, ax = plt.subplots(len(HORIZONS), 1, figsize=(11, 9), sharex=True, sharey=True)
for a, h, cible in zip(ax, HORIZONS, Y.columns):
    dates = grille + pd.Timedelta(hours=12 * h)           # date visée par la prédiction
    a.plot(dates, Y_test[cible].reindex(grille), color='k', lw=1.2, label='observé')
    for nom, trait in TRAIT.items():
        a.plot(dates, predictions[nom][cible].reindex(grille), color=COULEUR[nom], ls=trait, lw=1, label=nom)
    a.set_ylabel(f'log10 flux\n{cible}')
ax[0].legend(fontsize=8, ncol=3, loc='upper left')
ax[0].set_title('Test 2024 : flux observé et flux prédit, pour chaque horizon')
fig.tight_layout()
savefig(fig, '8_predit_temps.png')
plt.show()

# Prédit en fonction d'observé : une ligne par modèle, une colonne par horizon
valeurs = pd.concat([Y_test.stack()] + [pred.stack() for pred in predictions.values()])
bornes = [valeurs.min() - 0.1, valeurs.max() + 0.1]
fig, ax = plt.subplots(len(predictions), len(Y.columns), figsize=(12, 8.5), sharex=True, sharey=True)
for i, (nom, pred) in enumerate(predictions.items()):
    for j, cible in enumerate(Y.columns):
        a = ax[i, j]
        a.plot(bornes, bornes, color='0.4', lw=1, ls='--')
        a.scatter(Y_test[cible], pred[cible], s=6, alpha=0.5, color=COULEUR[nom], linewidths=0)
        a.text(0.05, 0.92, f"R² = {resultats[nom].loc[cible, 'R²']:.2f}", transform=a.transAxes, fontsize=9)
        a.set_xlim(bornes)
        a.set_ylim(bornes)
        a.set_aspect('equal')
        if i == 0:
            a.set_title(cible)
        if i == len(predictions) - 1:
            a.set_xlabel('log10 flux observé')
    ax[i, 0].set_ylabel(f'{nom}\nlog10 flux prédit')
fig.tight_layout()
savefig(fig, '8_predit_observe.png')
plt.show()

# Erreur sur les grandes variations (facteur 3 ou plus entre t et t + h)
GRANDE_VARIATION = 0.5
variation = Y_test.sub(X_test['log_flux_t-0h'], axis=0).abs() > GRANDE_VARIATION
rmse_var = pd.DataFrame({nom: {h: scores(Y_test.loc[variation[h], h], pred.loc[variation[h], h])['RMSE'] for h in Y.columns}
                         for nom, pred in predictions.items()})
rmse_var.insert(0, 'fenêtres', variation.sum())
display(rmse_var.round(3))

**Diagnostic : pourquoi la Random Forest fait-elle moins bien que la régression linéaire ?**
Deux explications sont testées :
1. **les flux très bas** : une forêt prédit une moyenne de valeurs vues à l'entraînement, elle ne peut pas prolonger
   une tendance. On compare donc les erreurs séparément pour les flux observés bas (log10 < 2,7) et les autres ;
2. **F10.7** : ce paramètre suit le cycle solaire et vaut 192 en moyenne en 2024, contre 110 sur 2020–2023. On
   réentraîne la même forêt **sans F10.7**.

⚠️ Ce diagnostic **utilise le test** : il sert à comprendre, pas à choisir un modèle. Garder la version sans F10.7
parce qu'elle est meilleure sur 2024 fausserait l'évaluation.

In [ ]:
def rmse_si(nom, h, masque):
    """RMSE du modèle `nom` à l'horizon h, sur les fenêtres de test sélectionnées par masque."""
    return np.sqrt(((Y_test[h] - predictions[nom][h])[masque] ** 2).mean())

FLUX_BAS = 2.7
LIN, RF = 'Régression linéaire', 'Random Forest'
sans_f107 = [c for c in X.columns if not c.startswith('F107')]
diagnostic = {}
for h in Y.columns:
    rf_sans = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=1 / 3,
                                    random_state=0, n_jobs=-1)
    rf_sans.fit(X_train[sans_f107], Y_train[h])
    bas = Y_test[h] < FLUX_BAS
    diagnostic[h] = {'fenêtres flux bas': bas.sum(),
                     'flux bas : linéaire': rmse_si(LIN, h, bas),
                     'flux bas : RF': rmse_si(RF, h, bas),
                     'autres flux : linéaire': rmse_si(LIN, h, ~bas),
                     'autres flux : RF': rmse_si(RF, h, ~bas),
                     'tous flux : linéaire': resultats[LIN].loc[h, 'RMSE'],
                     'tous flux : RF': resultats[RF].loc[h, 'RMSE'],
                     'tous flux : RF sans F10.7': scores(Y_test[h], rf_sans.predict(X_test[sans_f107]))['RMSE']}
display(pd.DataFrame(diagnostic).T.round(3))
print('Plus petite prédiction de la RF :', predictions[RF].min().round(2).to_dict())
print('Plus petite prédiction linéaire  :', predictions[LIN].min().round(2).to_dict())
print(f'Plus petit flux observé : test {Y_test.min().min():.2f}, entraînement {Y_train.min().min():.2f}')

**Lecture :**
* **les modèles suivent bien la tendance** (figure au fil du temps), et les nuages de points s'élargissent avec
  l'horizon, comme le disent les métriques ;
* **les chutes brutales (dropouts) ne sont pas anticipées** : les creux noirs d'avril, de septembre et de novembre 2024,
  par exemple, sont absents des prédictions à +24 h et au-delà ;
* **sur les grandes variations** (facteur 3 ou plus), les deux modèles réduisent nettement l'erreur de la persistance
  (RMSE 0,41–0,45 contre 0,68–0,73), mais elle reste d'un **facteur 2,5 à 3**. Ces cas sont peu nombreux (6 à 53
  fenêtres selon l'horizon) et choisis d'après ce qui s'est réellement passé : c'est une analyse, pas une prédiction ;
* la régression linéaire produit quelques **prédictions aberrantes vers le bas** (jusqu'à log10 = 1,87, alors que le
  plus petit flux observé vaut 1,94).

**Pourquoi la Random Forest est derrière la régression linéaire :**
* **son problème se concentre sur les flux bas** (environ 25 % du test) : sous log10 = 2,7, sa RMSE atteint 0,22 à 0,40
  contre 0,17 à 0,24 pour la régression. Sur les autres fenêtres, **elle est meilleure** à tous les horizons (à +48 h :
  0,209 contre 0,214). Sa plus petite prédiction reste à 2,34–2,48, alors que le flux descend à 1,94 début 2024 (2,18 au
  minimum à l'entraînement) ;
* **F10.7 la pénalise sur 2024** : sans ce paramètre, sa RMSE passe de 0,268 à 0,234 à +48 h, et de 0,163 à 0,151 à
  +12 h. Elle se rapproche ainsi de la régression linéaire sans la rattraper à +24 h et au-delà (0,234 contre 0,221 à
  +48 h), et la dépasse de peu à +12 h. Interprétation probable, non démontrée : en 2024, F10.7 est élevé alors que le
  flux est très bas, une combinaison rare à l'entraînement.

Le modèle retenu pour la comparaison reste la Random Forest entraînée plus haut, réglée sans regarder le test. Pour vraiment
retirer F10.7, il faudrait décider sur une période de validation prise dans l'entraînement (par exemple apprendre sur
2020–2022 et valider sur 2023).